In [59]:
# Extra info for deleting model/pipeline objects and freeing up GPU memory
# import gc

# # Delete the objects holding references to the model/pipeline
# del llm, chat  # add any other model/pipeline variables you created

# gc.collect()          # clear Python-level references
# torch.cuda.empty_cache()      # release cached (but unused) VRAM back to the OS
# torch.cuda.ipc_collect()      # clean up any inter-process memory
%pip install langchain_cohere

Note: you may need to restart the kernel to use updated packages.


In [60]:
from langchain_classic.retrievers import EnsembleRetriever # Package name moved from langchain.retrievers to langchain_classic.retrievers
from langchain_community.retrievers import BM25Retriever # Package name moved from langchain.retrievers to langchain_community.retrievers
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_cohere import CohereRerank
from langchain_core.messages import HumanMessage, SystemMessage
import os

from config.config import DEVICE, DB_PATH, EMBEDDING_MODEL_NAME, EMBEDDING_KWARGS
import torch

from dotenv import load_dotenv

load_dotenv() 

True

In [61]:
chunks = [
    # Tesla - Financial & Production
    "Tesla reported record quarterly revenue of $25.2 billion in Q3 2024.",
    "Tesla's automotive gross margin improved to 19.3% this quarter.",
    "Tesla Cybertruck production ramp begins in 2024 with initial deliveries.",
    "Tesla announced plans to expand Gigafactory production capacity.",
    "Tesla stock price reached new highs following earnings announcement.",
    "Tesla's energy storage business grew 40% year-over-year.",
    "Tesla continues to lead in electric vehicle market share globally.",
    "Tesla Model Y became the best-selling vehicle worldwide.",
    "Tesla reported strong free cash flow generation of $7.5 billion.",
    "Tesla's Full Self-Driving revenue increased significantly.",
    
    # Microsoft - Development & Acquisitions
    "Microsoft acquired GitHub for $7.5 billion in 2018.",
    "Microsoft's cloud revenue Azure grew 29% year-over-year.",
    "Microsoft announced new AI features for Visual Studio Code.",
    "Microsoft Teams integration with GitHub enhances developer workflow.",
    "Microsoft's developer tools division sees strong adoption.",
    "Microsoft acquired Activision Blizzard for $68.7 billion.",
    "Microsoft's productivity suite gained 50 million new users.",
    "Microsoft announced new Surface devices for developers.",
    "Microsoft's AI Copilot features expand to more development tools.",
    "Microsoft's enterprise solutions drive revenue growth.",
    
    # NVIDIA - AI & Hardware
    "NVIDIA's data center revenue reached $47.5 billion annually.",
    "NVIDIA's H100 GPUs see unprecedented demand for AI training.",
    "NVIDIA announced next-generation Blackwell architecture.",
    "NVIDIA's gaming revenue declined due to crypto market changes.",
    "NVIDIA's automotive AI platform partnerships expanded.",
    "NVIDIA's AI chip shortage affects cloud providers.",
    "NVIDIA stock valuation exceeds $2 trillion market cap.",
    "NVIDIA's CUDA platform dominates AI development.",
    "NVIDIA announced new AI inference chips for edge computing.",
    "NVIDIA's partnership with major cloud providers strengthens.",
    
    # Google/Alphabet - AI & Cloud
    "Google's AI investments total over $100 billion in recent years.",
    "Google Cloud revenue grew 35% reaching $8.4 billion quarterly.",
    "Google announced Gemini AI model competing with GPT-4.",
    "Google's search advertising revenue remains strong at $59 billion.",
    "Google's Workspace products integrate advanced AI features.",
    "Google announced quantum computing breakthroughs.",
    "Google's autonomous vehicle division Waymo expands operations.",
    "Google's AI research published breakthrough papers.",
    "Google's cloud AI services see enterprise adoption.",
    "Google faces regulatory scrutiny over AI dominance.",
    
    # Noisy/Less Relevant Chunks
    "The Tesla coil was invented by Nikola Tesla in 1891.",
    "Microsoft Excel spreadsheet formulas can be complex for beginners.",
    "NVIDIA Shield TV streaming device gets software update.",
    "Google Maps navigation improved with real-time traffic data.",
    "Production delays affected multiple manufacturing sectors.",
    "Financial markets showed volatility during earnings season.",
    "Revenue recognition standards changed for software companies.",
    "Hardware components face supply chain constraints globally.",
    "Development tools market grows with remote work trends.",
    "AI research requires significant computational resources.",
    "Quarterly reports show mixed results across tech sector.",
    "Stock market analysts upgrade technology sector ratings.",
    "Cloud computing adoption accelerates in enterprise market.",
    "Data center construction increases globally.",
    "Semiconductor shortage impacts various industries.",
    "Electric vehicle charging infrastructure expands rapidly.",
    "Software development productivity tools gain popularity.",
    "Machine learning frameworks become more accessible.",
    "Enterprise software licensing models evolve.",
    "Technology conferences showcase latest innovations."
]

print(f"Created {len(chunks)} sample chunks for demonstration")

Created 60 sample chunks for demonstration


In [62]:
# Convert to Document objects
documents = [Document(page_content=chunk, metadata={"source": f"chunk_{i}"}) for i, chunk in enumerate(chunks)]

# 1. Vector Retriever (Semantic Search/Dense Retrieval)

In [63]:
print("Setting up Vector retriever with HuggingFace Embeddings and Chroma vector store...")

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": DEVICE},
    encode_kwargs=EMBEDDING_KWARGS,
)

vector_db = Chroma(
    persist_directory=DB_PATH,
    embedding_function=embedding_model,
    collection_metadata={"hnsw:space": "cosine"},
)

vector_retriever = vector_db.as_retriever(search_kwargs={"k": 2})

Setting up Vector retriever with HuggingFace Embeddings and Chroma vector store...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19321.88it/s]


# 2. BM25 Retriever (Keyword Search/Sparse Retrieval)

In [64]:
print("Setting up BM25 retriever for keyword search...")
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 3  # Set the number of documents to retrieve

Setting up BM25 retriever for keyword search...


# 3. Hybrid Retriever

In [65]:
# Hybrid search: Combine BM25 and vector search results
print("\nSetting up Hybrid Retriever...")
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.7, 0.3]  # Adjust weights as needed for your use case
)


Setting up Hybrid Retriever...


In [66]:
query = "Tesla financial performance and production updates"

print("STEP 1: Hybrid Search Results")
print("-"*50)

retrieved_docs = hybrid_retriever.invoke(query)  # Get top 25 for reranking

# Show top 10 from hybrid search
for i, doc in enumerate(retrieved_docs, 1):
    print(f"{i:2d}. {doc.page_content}")

print(f"\n(Retrieved {len(retrieved_docs)} total chunks for reranking)\n")

STEP 1: Hybrid Search Results
--------------------------------------------------
 1. === Production and sales by quarter ===

Tesla deliveries vary significantly by month due to regional issues such as availability of car carriers and registration. On March 9, 2020, the company produced its 1 millionth electric car, becoming the first auto manufacturer to achieve such a milestone. In the third quarter of 2021, Tesla sold its 2 millionth electric car, becoming the first auto manufacturer to achieve such a milestone. In the first quarter of 2023, the Model Y became the world's best-selling car, surpassing the Toyota Corolla.


== Finances ==

For the fiscal (and calendar) year 2021, Tesla reported a net income of $5.52 billion. The annual revenue was $53.8 billion, an increase of 71% over the previous fiscal year.
 2. === Global expansion and Model Y (2019–present) ===
From July 2019 to June 2020, Tesla reported four consecutive profitable quarters for the first time, which made it eligi

In [ ]:
print("STEP 2: After Cohere Reranking (Top 10)")
print("-"*50)

# Initialize Cohere reranker
reranker = CohereRerank(model="rerank-english-v3.0", top_n=10) # Cohere Rerank model is used to rerank the retrieved documents based on their relevance to the query. Using API key from environment variable COHERE_API_KEY.

# Rerank the retrieved documents
reranked_docs = reranker.compress_documents(retrieved_docs, query)

# Show reranked results
for i, doc in enumerate(reranked_docs, 1):
    print(f"{i:2d}. {doc.page_content}")

print("\n" + "="*80)
print("ANALYSIS:")
print("✅ Hybrid Search: Mixed relevant and irrelevant results")
print("✅ Reranking: Most relevant Tesla financial/production info at top")
print("✅ Notice how reranking moved the most contextually relevant chunks higher")

# Optional: Show the difference more clearly
print("\n" + "="*80)
print("KEY IMPROVEMENTS AFTER RERANKING:")
print("-"*40)


hybrid_top_5 = [doc.page_content for doc in retrieved_docs[:5]]
reranked_top_5= [doc.page_content for doc in reranked_docs[:5]]

print("BEFORE (Hybrid Top 3):")
for i, content in enumerate(hybrid_top_5, 1):
    print(f"  {i}. {content}")

print("\nAFTER (Reranked Top 3):")
for i, content in enumerate(reranked_top_5, 1):
    print(f"  {i}. {content}")

STEP 2: After Cohere Reranking (Top 10)
--------------------------------------------------
 1. === Production and sales by quarter ===

Tesla deliveries vary significantly by month due to regional issues such as availability of car carriers and registration. On March 9, 2020, the company produced its 1 millionth electric car, becoming the first auto manufacturer to achieve such a milestone. In the third quarter of 2021, Tesla sold its 2 millionth electric car, becoming the first auto manufacturer to achieve such a milestone. In the first quarter of 2023, the Model Y became the world's best-selling car, surpassing the Toyota Corolla.


== Finances ==

For the fiscal (and calendar) year 2021, Tesla reported a net income of $5.52 billion. The annual revenue was $53.8 billion, an increase of 71% over the previous fiscal year.
 2. === Global expansion and Model Y (2019–present) ===
From July 2019 to June 2020, Tesla reported four consecutive profitable quarters for the first time, which mad

In [68]:

print("\n" + "="*80)
print("FINAL: RAG with Reranked Context")
print("-"*40)

# Use top 5 reranked documents for final answer
top_reranked = reranked_docs[:5]

combined_input = f"""Based on the following documents, please answer this question: {query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in top_reranked])}

Please provide a clear, helpful answer using only the information from these documents."""

# Set up the HuggingFace model for text generation
llm = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    device_map="auto",
    model_kwargs={"dtype": torch.float16},          # load weights in fp16 to fit VRAM
    pipeline_kwargs={
        "temperature": 0.2,
        "do_sample": True,          # required for temperature to have any effect
        "max_new_tokens": 512,      # explicit cap, avoids conflicting with the model's default max_length=20
        "return_full_text": False,  # return ONLY the generated answer, not prompt+answer glued together
    },
)

chat = ChatHuggingFace(llm=llm)

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=combined_input),
]

result = chat.invoke(messages)
print("Generated Response:")
print(result.content)


FINAL: RAG with Reranked Context
----------------------------------------


Loading weights: 100%|██████████| 339/339 [00:03<00:00, 88.17it/s] 
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corru

Generated Response:
Based on the provided documents, here are the key financial and production updates for Tesla:

### Financial Performance:
- **Net Income for 2021:** Tesla reported a net income of $5.52 billion for the fiscal (and calendar) year 2021.
- **Revenue Growth:** The annual revenue was $53.8 billion, marking a 71% increase over the previous fiscal year.
- **Market Capitalization:** By December 14, 2020, Tesla's market capitalization exceeded that of the next nine largest automakers combined, making it the sixth most valuable company in the U.S. It reached $1 trillion in market capitalization in October 2021.

### Production Updates:
- **Model Y Sales:** In the first quarter of 2023, the Model Y became the world's best-selling car, surpassing the Toyota Corolla.
- **Production Milestones:**
  - On March 9, 2020, Tesla produced its 1 millionth electric car, becoming the first auto manufacturer to achieve this milestone.
  - In the third quarter of 2021, Tesla sold its 2 mill